In [5]:
import casadi as ca
import numpy as np

In [6]:
# Données du problème
l, m, mr, md = 7, 5, 2, 3

A = np.array([[-1, -1, 0, 0, 0, 0, 0],
              [0, 0, -1, -1, 0, 0, 0],
              [1, 0, 0, 1, -1, 0, -1],
              [0, 1, 0, 0, 1, -1, 0],
              [0, 0, 1, 0, 0, 1, 1]])

r = np.array([100, 10, 1000, 100, 100, 10, 1000])
pr = np.array([105, 104])
fd = np.array([0.08, -1.3, 0.13])

In [ ]:
# Variables de décision
q = ca.SX.sym('q', l)
p = ca.SX.sym('p', m)
f = ca.SX.sym('f', m)

# Fonction objectif
obj = ca.sumsqr(A @ q - f) + ca.sumsqr(r * q * ca.fabs(q) + A.T @ p)

# Contraintes d'égalité
eq1 = A @ q - f
eq2 = A.T @ p + r * q * ca.fabs(q)
# Contraintes sur les pressions imposées
c_pr = ca.vertcat(p[:mr] - pr)
# Contraintes sur les flux imposés
c_fd = ca.vertcat(f[mr:] - fd)

# Optimisation
nlp = {'x': ca.vertcat(q, p, f), 'f': obj,
       'g': ca.vertcat(eq1, eq2, c_pr, c_fd)}

solver = ca.nlpsol('solver', 'ipopt', nlp)

# Résolution
res = solver(x0=np.zeros(l + m + m),
             lbx=-ca.inf,
             ubx=ca.inf,
             lbg=0,
             ubg=0)

This is Ipopt version 3.14.11, running with linear solver MUMPS 5.4.1.

Number of nonzeros in equality constraint Jacobian...:       45
Number of nonzeros in inequality constraint Jacobian.:        0
Number of nonzeros in Lagrangian Hessian.............:       66

Total number of variables............................:       17
                     variables with only lower bounds:        0
                variables with lower and upper bounds:        0
                     variables with only upper bounds:        0
Total number of equality constraints.................:       17
Total number of inequality constraints...............:        0
        inequality constraints with only lower bounds:        0
   inequality constraints with lower and upper bounds:        0
        inequality constraints with only upper bounds:        0

iter    objective    inf_pr   inf_du lg(mu)  ||d||  lg(rg) alpha_du alpha_pr  ls
   0  0.0000000e+00 1.05e+02 0.00e+00  -1.0 0.00e+00    -  0.00e+00 0.00e+00 

In [10]:
# Extraction des solutions
q_opt = np.array(res['x'][:l].full().flatten())
p_opt = np.array(res['x'][l:l+m].full().flatten())
f_opt = np.array(res['x'][l+m:].full().flatten())

# Vérification des contraintes
f = A @ q_opt
Kirchoff = A.T @ p_opt + r * q_opt * np.abs(q_opt)

print("Débits optimaux q:", q_opt)
print("Pressions optimales p:", p_opt)
print("Bilan de flux (f):", f)
print("Loi de Kirchoff (A^T p + r • q • |q|):", Kirchoff)

Débits optimaux q: [-0.08814896 -0.78830544 -0.08024054 -0.13330506 -0.2331787   0.27851586
 -0.06827532]
Pressions optimales p: [105.         104.         105.77702396 111.21425464 110.43854379]
Bilan de flux (f): [ 0.8764544  0.2135456  0.08      -1.3        0.13     ]
Loi de Kirchoff (A^T p + r • q • |q|): [-3.27227134e-12 -6.38156195e-12 -1.22003563e-09 -1.55431223e-12
 -4.14601686e-12  1.00535136e-11 -1.04058984e-11]
